# 4단계: 내 녹음으로 오경보 평가 + 학습에 추가해서 전/후 비교

드라이브의 `6_own`(학습용·시험용 녹음)을 읽어서 다음을 합니다.
1. 녹음을 16kHz 모노로 바꾸고 YAMNet 임베딩을 뽑는다. (말소리 비율도 같이 확인)
2. 현재 모델(03, 03b)이 **시험용 녹음에서 시간당 몇 번 오경보를 내는지** 잰다. 실제 연속 소리처럼 "N프레임 연속 + 쉬는 시간 5초" 규칙을 쓴다.
3. 학습용 녹음(정상 소리)을 학습에 추가해서 다시 학습하고, 같은 시험용으로 **전/후**를 비교한다.

런타임은 **CPU로도 됩니다**(YAMNet 단계가 몇 분 더 걸릴 뿐). GPU가 잡혀 있으면 더 빠릅니다.
위에서부터 순서대로 실행하고, **3번, 6번, 7번 항목의 출력**을 복사해서 알려 주세요.

## 0. 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 파일 찾기

In [ ]:
import glob, os, re, subprocess, pickle, time, csv, unicodedata
import numpy as np
import pandas as pd

def find(name):
    hits = glob.glob(f"/content/drive/MyDrive/**/{name}", recursive=True)
    assert hits, f"{name}을 못 찾았어요. 드라이브에 올렸는지 확인하세요."
    return hits[0]

OWN = find("6_own")
EMB = find("embeddings_v2.npz")
OUT_DIR = os.path.dirname(EMB)
files = sorted(glob.glob(f"{OWN}/*/*.m4a") + glob.glob(f"{OWN}/*/*.wav") + glob.glob(f"{OWN}/*/*.mp3"))
print("녹음 파일", len(files), "개 / 작업 폴더:", OUT_DIR)

## 2. YAMNet 불러오기

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
with tf.io.gfile.GFile(yamnet.class_map_path().numpy()) as f:
    names = [row["display_name"] for row in csv.DictReader(f)]
SPEECH = names.index("Speech")
SCREAM = [names.index(n) for n in ("Screaming", "Shout", "Yell")]
print(len(names), "클래스")

## 3. 녹음 변환과 임베딩 (몇 분 걸림)
- 스테레오는 두 채널을 평균하되, 좌우 상관이 -0.3보다 낮으면(위상 반대) 더 큰 채널 하나만 씁니다.
- 파일을 자르지 않고 **연속으로** 처리합니다. 프레임 간격은 약 0.48초.
- 출력의 말소리 비율은 YAMNet이 "말소리"라고 본 프레임의 비율입니다. 대화·통화·TV는 높고 조용한 방은 낮아야 합니다.

In [ ]:
FFMPEG = "ffmpeg"
SR, HOP, WIN = 16000, 7680, 15600
SEG = HOP * 600          # 288초씩 나눠서 처리 (프레임 격자는 이어짐)

def probe_channels(path):
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", path], capture_output=True, text=True, errors="replace")
    m = re.search(r"Audio: .*?, \d+ Hz, ([^,\n]+)", r.stderr)
    return 1 if (m and m.group(1).strip().startswith(("mono", "1 channel"))) else 2

def load_mono(path):
    ch = probe_channels(path)
    raw = subprocess.run([FFMPEG, "-v", "error", "-i", path, "-vn", "-ac", str(ch), "-ar", str(SR), "-f", "s16le", "-"],
                         capture_output=True).stdout
    x = np.frombuffer(raw[: len(raw) // (2 * ch) * (2 * ch)], dtype=np.int16).reshape(-1, ch)
    if ch == 1:
        return x[:, 0].copy(), "모노", None
    a, b = x[::10, 0].astype(np.float64), x[::10, 1].astype(np.float64)
    corr = float(np.corrcoef(a, b)[0, 1]) if a.std() > 0 and b.std() > 0 else 1.0
    if corr < -0.3:
        return x[:, int(b.std() > a.std())].copy(), "한 채널(위상 반대)", corr
    return x.astype(np.float32).mean(axis=1).astype(np.int16), "두 채널 평균", corr

def embed_stream(x):
    embs, sp, sc, db, ts = [], [], [], [], []
    n = len(x)
    for start in range(0, n, SEG):
        seg = x[start:start + SEG + (WIN - HOP)]
        if len(seg) < WIN:
            break
        w = seg.astype(np.float32) / 32768
        scores, emb, _ = yamnet(tf.constant(w))
        scores, emb = scores.numpy(), emb.numpy()
        last = start + SEG + (WIN - HOP) >= n
        nb = len(w) // HOP
        bp = (w[: nb * HOP].reshape(nb, HOP) ** 2).mean(axis=1)
        fdb = 10 * np.log10(((bp[:-1] + bp[1:]) / 2 if nb > 1 else bp) + 1e-12)
        k = min(len(emb) if last else 600, len(emb), len(fdb))
        embs.append(emb[:k].astype(np.float16))
        sp.append(scores[:k, SPEECH].astype(np.float16))
        sc.append(scores[:k][:, SCREAM].max(axis=1).astype(np.float16))
        db.append(fdb[:k].astype(np.float32))
        ts.append(start / SR + np.arange(k) * 0.48)
    return tuple(np.concatenate(a) for a in (embs, sp, sc, db, ts))

own, t0 = {}, time.time()
for f in files:
    x, how, corr = load_mono(f)
    emb, sp, sc, db, ts = embed_stream(x)
    nfc = lambda s: unicodedata.normalize("NFC", s)   # 드라이브의 한글 이름은 자모가 풀어진 형태일 수 있음
    own[f] = dict(split=nfc(os.path.basename(os.path.dirname(f))), name=nfc(os.path.basename(f)), emb=emb, speech=sp,
                  scream=sc, db=db, t=ts, dur=len(x) / SR, how=how, corr=corr)
    print(f"{own[f]['split']}/{own[f]['name'][:28]:<28} {own[f]['dur']/60:5.1f}분 | {how}"
          f"{'' if corr is None else f' (상관 {corr:.2f})'} | 말소리 {float((sp > 0.5).mean()):.0%} | "
          f"비명류(>0.3) {float((sc > 0.3).mean()):.0%}", flush=True)
pickle.dump(own, open(f"{OUT_DIR}/own_embeddings_v1.pkl", "wb"))
print(f"\n완료 ({(time.time() - t0) / 60:.1f}분), 저장: own_embeddings_v1.pkl")

## 4. 평가 도구와 AI Hub 임베딩 읽기

In [ ]:
from tensorflow import keras

d = np.load(EMB)
X, S, clip_idx = d["X"].astype(np.float32), d["speech"].astype(np.float32), d["clip_idx"]
df = pd.read_csv(find("dataset_v2.csv"), encoding="utf-8-sig")
y_clip = (df["label"] == "urgent").astype(int).values
is_tr = (df["split"] == "train").values
starts = np.r_[0, np.flatnonzero(np.diff(clip_idx)) + 1]
ends = np.r_[starts[1:], len(clip_idx)]
frames = ends - starts
va_urgent_idx = np.flatnonzero((~is_tr) & (y_clip == 1))

test_files = [f for f, v in own.items() if v["split"] == "시험용" and "위급 장면" not in v["name"]]
scene_files = [f for f, v in own.items() if "위급 장면" in v["name"]]
assert test_files, "시험용 녹음을 못 찾았어요. 폴더 이름(학습용/시험용)을 확인하세요."
hours = sum(own[f]["dur"] for f in test_files) / 3600
print(f"시험용 일반 소리 {hours * 60:.0f}분 (TV 위급 장면 파일 {len(scene_files)}개는 따로 계산)")

def clip_scores(p, N):
    out = []
    for i in va_urgent_idx:
        q = p[starts[i]:ends[i]]
        out.append(np.lib.stride_tricks.sliding_window_view(q, N).min(axis=1).max() if len(q) >= N else 0.0)
    return np.array(out)

def alarm_idx(p, t, N, thr, cooldown=5.0):
    ok = (p >= thr).astype(int)
    if len(ok) < N:
        return []
    run = np.convolve(ok, np.ones(N, int), "valid") == N
    out, last = [], -1e9
    for i in np.flatnonzero(run) + (N - 1):
        if t[i] - last >= cooldown:
            out.append(int(i)); last = t[i]
    return out

def tradeoff(model, name):
    p_ai = model.predict(X, batch_size=4096, verbose=0).ravel()
    p_own = {f: model.predict(v["emb"].astype(np.float32), batch_size=4096, verbose=0).ravel() for f, v in own.items()}
    rows = []
    for N in (1, 2, 3):
        sc = clip_scores(p_ai, N)
        for thr in (0.5, 0.8, 0.9, 0.95):
            n_a = sum(len(alarm_idx(p_own[f], own[f]["t"], N, thr)) for f in test_files)
            n_s = sum(len(alarm_idx(p_own[f], own[f]["t"], N, thr)) for f in scene_files)
            rows.append({"모델": name, "N": N, "임계값": thr, "AIHub위급재현율": (sc >= thr).mean(),
                         "시험용오경보(횟수)": n_a, "시간당오경보": n_a / hours, "TV위급장면알림": n_s})
    return pd.DataFrame(rows), p_own

before = []
for fn, label in (("classifier_mlp_v1.keras", "03"), ("classifier_mlp_v2.keras", "03b")):
    try:
        m = keras.models.load_model(find(fn))
    except AssertionError as e:
        print("건너뜀:", e); continue
    t, _ = tradeoff(m, label)
    before.append(t)
print("현재 모델 평가 완료:", len(before), "개")

## 5. 내 녹음(학습용)을 정상 소리로 추가해서 다시 학습
- 학습용 녹음을 약 10초 조각으로 나눠 **정상 클립**으로 넣습니다. (`OWN_BOOST`배 가중치)
- 나머지 설정은 03b와 같습니다 (정상 3배, 말소리 없는 위급 프레임 제외).
- 시험용 녹음은 학습에 쓰지 않습니다.

In [ ]:
OWN_BOOST, NORMAL_BOOST, SPEECH_MIN, CH = 3.0, 3.0, 0.2, 20
chunks = [v["emb"][k * CH:(k + 1) * CH] for v in own.values() if v["split"] == "학습용" for k in range(len(v["emb"]) // CH)]
n_own = len(chunks)
print("학습용 녹음 조각:", n_own, "개 (약 10초씩)")

X_all = np.concatenate([X, np.concatenate(chunks).astype(np.float32)])
S_all = np.concatenate([S, np.ones(n_own * CH, np.float32)])
clip_all = np.concatenate([clip_idx, len(df) + np.repeat(np.arange(n_own), CH)])
y_all = np.r_[y_clip, np.zeros(n_own, int)]
tr_all = np.r_[is_tr, np.ones(n_own, bool)]
fr_all = np.bincount(clip_all)
boost = np.r_[np.ones(len(df)), np.full(n_own, OWN_BOOST)]

n_urg = int(y_all[tr_all].sum()); n_nor = int(tr_all.sum()) - n_urg
cw = {1: 0.5 * tr_all.sum() / n_urg, 0: NORMAL_BOOST * 0.5 * tr_all.sum() / n_nor}
w = np.array([cw[int(l)] for l in y_all])[clip_all] * boost[clip_all] / fr_all[clip_all]
y_f = y_all[clip_all].astype(np.float32)
tr_f = tr_all[clip_all]
w[(y_f == 1) & (S_all < SPEECH_MIN)] = 0
w = (w / w[tr_f].mean()).astype(np.float32)

tf.keras.utils.set_random_seed(42)
model_own = keras.Sequential([
    keras.layers.Input(shape=(1024,)),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model_own.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy")
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
model_own.fit(X_all[tr_f], y_f[tr_f], sample_weight=w[tr_f],
              validation_data=(X_all[~tr_f], y_f[~tr_f], w[~tr_f]),
              epochs=40, batch_size=512, callbacks=[es], verbose=2)

## 6. 전/후 비교
- **AIHub위급재현율**: AI Hub Validation 위급 클립 중 알림이 울린 비율
- **시간당오경보**: 시험용 일반 소리(대화·통화·TV·낭독·생활 소음·조용한 방)에서 울린 알림 횟수 ÷ 시간 (쉬는 시간 5초)
- **TV위급장면알림**: "TV 싸움, 응급, 위급 장면" 파일에서 울린 횟수 (참고용, 오경보 계산에서 제외)
- 모델 `자기녹음추가`가 학습용 녹음을 더한 새 모델입니다.

In [ ]:
after, p_after = tradeoff(model_own, "자기녹음추가")
pd.set_option("display.width", 200)
table = pd.concat(before + [after], ignore_index=True)
print(table.round(3).to_string(index=False))

## 7. 오경보가 울린 시각 (직접 들어 보기용)
`SHOW`의 (N, 임계값)에서 새 모델이 울린 시각입니다. 영상 플레이어로 그 시각(분:초)을 열어서 **무슨 소리였는지** 들어 보세요.

In [ ]:
SHOW = (2, 0.9)
mmss = lambda s: f"{int(s) // 60}:{int(s) % 60:02d}"
N0, THR0 = SHOW
print(f"N={N0}, 임계값={THR0}, 쉬는 시간 5초")
for f in test_files + scene_files:
    v, p = own[f], p_after[f]
    idx = alarm_idx(p, v["t"], N0, THR0)
    print(f"\n[{v['name']}]{' (TV 위급 장면, 참고)' if f in scene_files else ''} 알림 {len(idx)}회 / {v['dur'] / 60:.1f}분")
    for i in idx[:10]:
        j = i - (N0 - 1)
        print(f"  {mmss(v['t'][j])}  확률 {p[j:i + 1].min():.2f}  말소리 {float(v['speech'][j]):.2f}  음량 {v['db'][j]:.0f}dBFS")

## 8. 모델 저장

In [ ]:
model_own.save(f"{OUT_DIR}/classifier_mlp_v3_own.keras")
print("저장 완료:", f"{OUT_DIR}/classifier_mlp_v3_own.keras")